# Baseline Revenue Forecast — Prophet + AutoARIMA

**Purpose**: Fit two complementary time-series models on historical quarterly revenue
and produce 4-quarter-ahead forecasts with 80% and 95% credible/confidence intervals.

**Models**:
- **Prophet** (Facebook/Meta): Bayesian structural time-series with additive seasonality
  and trend changepoints. Suitable for business time series with seasonality.
- **AutoARIMA** (statsforecast): Automatic ARIMA order selection via information
  criterion (AIC). Widely used in quantitative finance.

**Honest disclaimer**: With ~20 quarterly observations, these models are underpowered
and their forecast intervals will be wide.  That is the correct outcome at this sample
size — not a bug.  We present both models to characterise the *range of plausible
outcomes*, not to claim point-forecast precision.

**Output**: `/models/{ticker}_baseline_forecasts.parquet`

---

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*cmdstan.*")

_cwd = Path(".").resolve()
REPO_ROOT = next(p for p in [_cwd, *_cwd.parents] if (p / "config" / "company.yaml").exists())
CONFIG_PATH = REPO_ROOT / "config" / "company.yaml"
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

with CONFIG_PATH.open() as fh:
    config = yaml.safe_load(fh)

TICKER = config["ticker"]
DB_PATH = REPO_ROOT / "data" / "processed" / f"{TICKER}.duckdb"

print(f"Ticker : {TICKER}")
print(f"DB     : {DB_PATH}")
print(f"Exists : {DB_PATH.exists()}")

## 1. Load Quarterly Revenue from DuckDB

In [ ]:
if not DB_PATH.exists():
    raise FileNotFoundError(
        f"DuckDB not found: {DB_PATH}\n"
        f"Run first: python -m src.build_warehouse --ticker {TICKER}"
    )

con = duckdb.connect(str(DB_PATH), read_only=True)
try:
    df_rev = con.execute("""
        SELECT period_end, fiscal_year, fiscal_period, Revenue
        FROM v_income_statement_quarterly
        WHERE period_type = 'Q'
          AND Revenue IS NOT NULL
        ORDER BY fiscal_year, fiscal_period
    """).fetchdf()
finally:
    con.close()

df_rev["period_end"] = pd.to_datetime(df_rev["period_end"])
df_rev = df_rev.sort_values("period_end").reset_index(drop=True)

n_obs = len(df_rev)
print(f"\nQuarterly Revenue observations: {n_obs}")
print(f'Date range: {df_rev["period_end"].min().date()} → {df_rev["period_end"].max().date()}')
print(
    f'Revenue range: ${df_rev["Revenue"].min() / 1e9:.2f}B – ${df_rev["Revenue"].max() / 1e9:.2f}B'
)
print()
display(df_rev[["period_end", "fiscal_year", "fiscal_period", "Revenue"]].tail(8))

## 2. Time-Series Cross-Validation Setup

We use **3 expanding-window folds** with a minimum training size of `floor(2/3 × n_obs)`.
Each fold trains on everything up to period T and evaluates the next 4 quarters.

Expanding windows are preferred over rolling windows at small n because:
- Every additional training observation is informative
- Rolling windows at small n can produce high-variance estimates

In [ ]:
N_FOLDS = 3
H = 4  # forecast horizon (quarters)
N_OBS = len(df_rev)

# Compute fold cut-points: evenly spaced from 2/3 to (n-H)
min_train = max(8, int(N_OBS * 2 / 3))
max_train = N_OBS - H

if min_train >= max_train:
    # Too few observations for 3 folds — use 1 fold
    fold_ends = [max_train]
else:
    fold_ends = [
        int(min_train + i * (max_train - min_train) / (N_FOLDS - 1)) for i in range(N_FOLDS)
    ]

print(f"Observations : {N_OBS}")
print(f"Folds        : {len(fold_ends)}")
print(f"Train sizes  : {fold_ends}")

for i, cut in enumerate(fold_ends):
    train_end = df_rev["period_end"].iloc[cut - 1].date()
    test_start = df_rev["period_end"].iloc[cut].date() if cut < N_OBS else "N/A"
    print(f"  Fold {i+1}: train → {train_end} | test starts {test_start}")

## 3. Prophet Forecast

**Configuration**:
- `yearly_seasonality=False` — quarterly data cannot identify intra-year seasonality
- `weekly_seasonality=False`, `daily_seasonality=False` — not applicable
- `changepoint_prior_scale=0.05` — modest flexibility; prevents overfitting at n≈20
- `seasonality_mode='multiplicative'` — appropriate for growing revenue series
- `interval_width=0.80` for 80% CI; re-run with 0.95 for 95% CI

Prophet requires columns `ds` (date) and `y` (value).

In [ ]:
from prophet import Prophet


def fit_prophet(train_df: pd.DataFrame, horizon: int, ci_level: float = 0.80) -> pd.DataFrame:
    """Fit Prophet on train_df and return forecast DataFrame.

    Args:
        train_df: DataFrame with columns [period_end, Revenue].
        horizon:  Number of quarterly periods to forecast.
        ci_level: Credible interval width (0.80 or 0.95).

    Returns:
        DataFrame with columns [ds, yhat, yhat_lower, yhat_upper] for forecast periods.
    """
    prophet_df = train_df.rename(columns={"period_end": "ds", "Revenue": "y"})[["ds", "y"]]

    m = Prophet(
        yearly_seasonality=False,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.05,
        seasonality_mode="multiplicative",
        interval_width=ci_level,
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        m.fit(prophet_df)

    # Build future dates — quarterly cadence (~91 days)
    last_date = train_df["period_end"].max()
    future_dates = pd.date_range(
        start=last_date + pd.DateOffset(months=3),
        periods=horizon,
        freq="QS",
    )
    future_df = pd.DataFrame({"ds": future_dates})
    forecast = m.predict(future_df)

    return forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()


# Cross-validation — Prophet
print("Running Prophet cross-validation...")
cv_results_prophet: list[dict] = []
for fold_idx, cut in enumerate(fold_ends):
    train = df_rev.iloc[:cut].copy()
    test = df_rev.iloc[cut : cut + H].copy()
    if len(test) == 0:
        continue

    fcst = fit_prophet(train, horizon=len(test), ci_level=0.80)
    actuals = test["Revenue"].values
    preds = fcst["yhat"].values[: len(actuals)]

    mae = float(np.mean(np.abs(preds - actuals)))
    mape = float(np.mean(np.abs((preds - actuals) / np.where(actuals > 0, actuals, np.nan))))
    cv_results_prophet.append({"fold": fold_idx + 1, "mae": mae, "mape": mape})
    print(f"  Fold {fold_idx + 1}  MAE=${mae/1e6:.0f}M  MAPE={mape:.1%}")

cv_prophet_df = pd.DataFrame(cv_results_prophet)
print(
    f'\nProphet avg  MAE=${cv_prophet_df["mae"].mean()/1e6:.0f}M  '
    f'MAPE={cv_prophet_df["mape"].mean():.1%}'
)

In [ ]:
# Fit final Prophet on all data — 80% and 95% CI
print("Fitting final Prophet model on full dataset...")
prophet_80 = fit_prophet(df_rev, horizon=H, ci_level=0.80)
prophet_95 = fit_prophet(df_rev, horizon=H, ci_level=0.95)

prophet_80.columns = ["period_end", "yhat", "yhat_lower_80", "yhat_upper_80"]
prophet_final = prophet_80.merge(
    prophet_95[["ds", "yhat_lower", "yhat_upper"]].rename(
        columns={"ds": "period_end", "yhat_lower": "yhat_lower_95", "yhat_upper": "yhat_upper_95"}
    ),
    on="period_end",
)
prophet_final.insert(0, "model", "prophet")
print(prophet_final.to_string(index=False))

## 4. AutoARIMA Forecast

**statsforecast.AutoARIMA** selects the optimal (p, d, q)(P, D, Q)_m order via AICc.

For quarterly data with ~20 observations:
- `season_length=4` — quarterly seasonality
- We limit max order to avoid overfitting: `max_p=2, max_q=2`
- Intervals are derived from residual variance (frequentist, not Bayesian)

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA


def fit_autoarima(train_df: pd.DataFrame, horizon: int) -> pd.DataFrame:
    """Fit AutoARIMA on train_df and return forecast DataFrame.

    Args:
        train_df: DataFrame with columns [period_end, Revenue].
        horizon:  Number of quarterly periods to forecast.

    Returns:
        DataFrame with yhat, lo-80, hi-80, lo-95, hi-95 for forecast periods.
    """
    sf_df = pd.DataFrame(
        {
            "unique_id": "revenue",
            "ds": train_df["period_end"].values,
            "y": train_df["Revenue"].values.astype(float),
        }
    )

    sf = StatsForecast(
        models=[AutoARIMA(season_length=4, max_p=2, max_q=2, alias="AutoARIMA")],
        freq="QS",
        n_jobs=1,
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fcst = sf.forecast(df=sf_df, h=horizon, level=[80, 95])

    return fcst.reset_index(drop=True)


# Cross-validation — AutoARIMA
print("Running AutoARIMA cross-validation...")
cv_results_arima: list[dict] = []
for fold_idx, cut in enumerate(fold_ends):
    train = df_rev.iloc[:cut].copy()
    test = df_rev.iloc[cut : cut + H].copy()
    if len(test) == 0:
        continue

    try:
        fcst = fit_autoarima(train, horizon=len(test))
        actuals = test["Revenue"].values
        preds = fcst["AutoARIMA"].values[: len(actuals)]

        mae = float(np.mean(np.abs(preds - actuals)))
        mape = float(np.mean(np.abs((preds - actuals) / np.where(actuals > 0, actuals, np.nan))))
        cv_results_arima.append({"fold": fold_idx + 1, "mae": mae, "mape": mape})
        print(f"  Fold {fold_idx + 1}  MAE=${mae/1e6:.0f}M  MAPE={mape:.1%}")
    except Exception as exc:
        print(f"  Fold {fold_idx + 1}  AutoARIMA error: {exc}")

if cv_results_arima:
    cv_arima_df = pd.DataFrame(cv_results_arima)
    print(
        f'\nAutoARIMA avg  MAE=${cv_arima_df["mae"].mean()/1e6:.0f}M  '
        f'MAPE={cv_arima_df["mape"].mean():.1%}'
    )

In [ ]:
# Fit final AutoARIMA on all data
print("Fitting final AutoARIMA model on full dataset...")
arima_raw = fit_autoarima(df_rev, horizon=H)

arima_final = pd.DataFrame(
    {
        "model": "autoarima",
        "period_end": arima_raw["ds"],
        "yhat": arima_raw["AutoARIMA"],
        "yhat_lower_80": arima_raw["AutoARIMA-lo-80"],
        "yhat_upper_80": arima_raw["AutoARIMA-hi-80"],
        "yhat_lower_95": arima_raw["AutoARIMA-lo-95"],
        "yhat_upper_95": arima_raw["AutoARIMA-hi-95"],
    }
)
print(arima_final.to_string(index=False))

## 5. Cross-Validation Summary

> **With ~20 quarterly observations, forecast intervals will be wide.
> That is the honest outcome at this sample size.**
>
> We present both models to characterise the range of plausible outcomes, not to
> claim that either model is statistically reliable. At n≈20, no single model is
> statistically defensible; Prompt 6 adds a Lasso macro model as a third perspective.

In [ ]:
summary_rows = []
for row in cv_results_prophet:
    summary_rows.append({"model": "Prophet", **row})
for row in cv_results_arima:
    summary_rows.append({"model": "AutoARIMA", **row})

cv_summary = pd.DataFrame(summary_rows)
if len(cv_summary) > 0:
    pivot = cv_summary.pivot_table(
        index="model",
        values=["mae", "mape"],
        aggfunc=["mean", "std"],
    )
    pivot.columns = ["MAE_mean", "MAPE_mean", "MAE_std", "MAPE_std"]
    pivot["MAE_mean_M"] = pivot["MAE_mean"] / 1e6
    display(pivot[["MAE_mean_M", "MAPE_mean"]].round(3))

## 6. Plot: Historical + Prophet + AutoARIMA Forecasts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
fig.suptitle(f"{TICKER} Quarterly Revenue — Baseline Forecasts", fontsize=13, fontweight="bold")

hist_dates = df_rev["period_end"]
hist_rev_b = df_rev["Revenue"] / 1e9  # billions

for ax, (model_name, fcst_df) in zip(
    axes,
    [("Prophet", prophet_final), ("AutoARIMA", arima_final)],
    strict=False,
):
    # Historical
    ax.plot(
        hist_dates,
        hist_rev_b,
        "o-",
        color="steelblue",
        linewidth=2,
        markersize=4,
        label="Historical",
        zorder=3,
    )

    fcst_dates = fcst_df["period_end"]
    yhat = fcst_df["yhat"] / 1e9
    lo_80 = fcst_df["yhat_lower_80"] / 1e9
    hi_80 = fcst_df["yhat_upper_80"] / 1e9
    lo_95 = fcst_df["yhat_lower_95"] / 1e9
    hi_95 = fcst_df["yhat_upper_95"] / 1e9

    # Bridge last historical point to forecast
    bridge_x = [hist_dates.iloc[-1], fcst_dates.iloc[0]]
    bridge_y = [hist_rev_b.iloc[-1], yhat.iloc[0]]
    ax.plot(bridge_x, bridge_y, "--", color="darkorange", linewidth=1.5, alpha=0.7)

    ax.fill_between(fcst_dates, lo_95, hi_95, alpha=0.15, color="darkorange", label="95% CI")
    ax.fill_between(fcst_dates, lo_80, hi_80, alpha=0.30, color="darkorange", label="80% CI")
    ax.plot(
        fcst_dates,
        yhat,
        "o--",
        color="darkorange",
        linewidth=2,
        markersize=5,
        label=f"{model_name} forecast",
        zorder=3,
    )

    ax.set_title(model_name, fontsize=11)
    ax.set_xlabel("Quarter End")
    ax.set_ylabel("Revenue ($B)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(
    REPO_ROOT / "dashboard" / "screenshots" / f"{TICKER}_baseline_forecast.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()
print("Saved forecast plot to dashboard/screenshots/")

## 7. Save Forecasts to Parquet

Output schema: `model, period_end, yhat, yhat_lower_80, yhat_upper_80, yhat_lower_95, yhat_upper_95`

In [ ]:
import sys

# Resolve src/ on the import path so we can use the vintage-stamping helper.
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.build_forecasts import write_forecast_parquet  # noqa: E402

combined = (
    pd.concat(
        [prophet_final, arima_final],
        ignore_index=True,
    )
    .sort_values(["model", "period_end"])
    .reset_index(drop=True)
)

# Validate schema
required_cols = [
    "model",
    "period_end",
    "yhat",
    "yhat_lower_80",
    "yhat_upper_80",
    "yhat_lower_95",
    "yhat_upper_95",
]
missing = [c for c in required_cols if c not in combined.columns]
assert not missing, f"Missing columns: {missing}"
assert combined["model"].isin(["prophet", "autoarima"]).all(), "Unexpected model names"
assert (combined["yhat"] > 0).all(), "Non-positive yhat values"

# write_forecast_parquet stamps every row with forecast_run_date (today UTC,
# ISO YYYY-MM-DD) — that's the model-training vintage that fact_forecasts.csv
# accumulates so re-runs produce a "forecasts I made vs how they landed" trail.
out_path = MODELS_DIR / f"{TICKER}_baseline_forecasts.parquet"
write_forecast_parquet(combined, out_path)

print(f"Saved {len(combined)} rows → {out_path}")
print("\nSchema:")
display(pd.read_parquet(out_path).dtypes.to_frame("dtype"))
print("\nForecast values (in $B):")
display(
    combined.assign(
        yhat_B=combined["yhat"] / 1e9,
        lo80_B=combined["yhat_lower_80"] / 1e9,
        hi80_B=combined["yhat_upper_80"] / 1e9,
    )[["model", "period_end", "yhat_B", "lo80_B", "hi80_B"]].to_string(index=False)
)

## 8. Interpretation Notes

### Why wide intervals are correct

At n≈20 quarterly observations:
- Prophet has limited data to identify trend slope and seasonality simultaneously
- AutoARIMA's residual variance estimates carry high uncertainty
- Confidence/credible intervals that *look* narrow at this sample size are almost
  certainly over-confident (a known failure mode of many time-series packages
  when n is small)

### What these models capture
- **Prophet**: trend momentum + structural breaks (changepoints). Good at capturing
  trend acceleration/deceleration visible in the historical series.
- **AutoARIMA**: mean-reversion and autocorrelation structure. Good at capturing
  quarter-to-quarter persistence.

### What these models miss
- Macro factors (interest rates, sector spending cycles) — addressed in Prompt 6 (Lasso)
- Management guidance / analyst consensus — addressed in Prompt 7 (yfinance)
- PANW-specific drivers: NGS ARR dynamics, platformization deal timing,
  Product vs. S&S mix shift — not available as structured XBRL data

### Using the output
The parquet file feeds into:
- Prompt 7: Tableau fact_forecasts.csv
- Prompt 7.5: v_variance_facts DuckDB view (actual vs. forecast)
- Prompt 8: LLM commentary (Claude reads pre-computed yhat values — does NOT recompute)